# Example Scenario Sweep: PV-BESS Sensitivity Analysis

This notebook demonstrates the complete workflow:
1. Load or generate hourly data (load, PV, prices)
2. Configure parameter sweeps (PV capacity, battery CAPEX, community size)
3. Run dispatch and billing for No-DER, PV-only, and PV-BESS scenarios
4. Calculate battery-specific value and economic metrics
5. Visualize results (sensitivity tables, payback periods, bill savings)

The simulation uses:
- Chapter 4 rule-based dispatch (dispatch_simulator.py)
- Spanish PVPC billing with member-level monthly cap (billing_calculator.py)
- Parameter sweep orchestrator (scenario_runner.py)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import thesis simulation modules
from dispatch_simulator import BESSDispatcher, PVOnlyDispatcher, NoDERDispatcher, DispatchScenario, BESSTechParams
from billing_calculator import BillingCalculator
from scenario_runner import ScenarioRunner, ScenarioConfig

%matplotlib inline
sns.set_style('whitegrid')

## 1. Generate or Load Hourly Data

For this example, we generate synthetic data for Seville, Spain (2026).
In practice, replace with real Silver layer Parquet files.

In [ ]:
# Generate synthetic hourly data for one year (2026)
np.random.seed(42)
T = 8760  # 365 days × 24 hours
timestamps = pd.date_range('2026-01-01', periods=T, freq='H', tz='Europe/Madrid')

# Synthetic load profile (30 households, daily/seasonal pattern)
# Base load with daily and seasonal modulation
hour_of_year = np.arange(T) / 24
hour_of_day = np.arange(T) % 24
day_of_year = np.arange(T) // 24

# Daily pattern: low at night, peaks in morning and evening
daily_pattern = 1.0 + 0.3 * np.sin((hour_of_day - 6) / 24 * 2 * np.pi) + 0.4 * np.sin((hour_of_day - 18) / 24 * 2 * np.pi)

# Seasonal pattern: higher in winter
seasonal_pattern = 1.1 + 0.2 * np.cos((day_of_year - 80) / 365 * 2 * np.pi)

# Baseline: 30 households, ~2.5 kWh per household per day
baseline_load_per_household = 2.5 / 24  # kWh per hour
load_kwh = np.maximum(0.3, baseline_load_per_household * 30 * daily_pattern * seasonal_pattern)

# PV generation (normalized to 1 kWp)
# Peak around noon, zero at night, seasonal variation
pv_daily = np.maximum(0, 3.5 * np.sin((hour_of_day - 6) / 12 * np.pi))
pv_seasonal = 1.0 + 0.3 * np.cos((day_of_year - 80) / 365 * 2 * np.pi)
pv_kwh_per_kwp = pv_daily * pv_seasonal

# OMIE prices (€/MWh): higher in evening, lower at night
omie_base = 50.0 + 30 * np.sin((hour_of_day - 6) / 24 * 2 * np.pi)
omie_seasonal = 1.0 + 0.15 * np.sin((day_of_year - 100) / 365 * 2 * np.pi)  # winter higher
omie_price = np.maximum(10, omie_base * omie_seasonal + np.random.normal(0, 5, T))

# PVPC import/export prices (simplified: energy-term only)
# Import prices follow OMIE with retail markup
import_price_eur_kwh = omie_price / 1000 * 1.5  # Convert €/MWh to €/kWh with markup
export_credit_eur_kwh = import_price_eur_kwh * 0.3  # Export is 30% of import price

print(f"Data generated: {T} hours")
print(f"Annual load: {load_kwh.sum():.0f} kWh")
print(f"Annual PV (1 kWp): {pv_kwh_per_kwp.sum():.0f} kWh")
print(f"Avg import price: €{import_price_eur_kwh.mean():.4f}/kWh")
print(f"Avg export credit: €{export_credit_eur_kwh.mean():.4f}/kWh")

# Visualize inputs
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(timestamps[:168], load_kwh[:168], 'b-', alpha=0.7)
axes[0, 0].set_title('Load (Week 1)')
axes[0, 0].set_ylabel('kWh/h')
axes[0, 1].plot(timestamps[:168], pv_kwh_per_kwp[:168], 'orange', alpha=0.7)
axes[0, 1].set_title('PV Generation 1 kWp (Week 1)')
axes[0, 1].set_ylabel('kWh/h')
axes[1, 0].hist(import_price_eur_kwh, bins=30, color='green', alpha=0.7)
axes[1, 0].set_title('Import Price Distribution')
axes[1, 0].set_xlabel('€/kWh')
axes[1, 1].hist(omie_price, bins=30, color='red', alpha=0.7)
axes[1, 1].set_title('OMIE Price Distribution')
axes[1, 1].set_xlabel('€/MWh')
plt.tight_layout()
plt.show()

## 2. Configure Scenario Sweep

Define parameter ranges for sensitivity analysis.

In [ ]:
# Create scenario config with parameter sweeps
config = ScenarioConfig(
    load_kwh=load_kwh,
    pv_kwh=pv_kwh_per_kwp,
    import_prices=import_price_eur_kwh,
    export_credits=export_credit_eur_kwh,
    omie_prices=omie_price,
    
    # Sensitivity parameters
    pv_capacities=[10, 15, 20, 25, 30],  # kWp
    battery_capex=[300, 400, 500, 600],  # €/kWh
    battery_capacity=150.0,  # kWh
    community_sizes=[30],  # household count
    cluster_compositions=[(0.4, 0.4, 0.2)],  # (low%, medium%, high%)
    
    # Economic assumptions
    omie_threshold_percentile=75.0,
    discount_rate=0.04,
    pv_lifetime=25,
    bess_lifetime=15,
    pv_capex=1100.0,  # €/kWp
)

# Create runner and preview scenarios
runner = ScenarioRunner(config)
print(f"Total scenarios to run: {len(runner.scenario_matrix)}")
for s in runner.scenario_matrix[:3]:
    print(f"  {s['scenario_id']}")
print(f"  ... ({len(runner.scenario_matrix) - 3} more)")
print()
print(f"Parameter space:")
print(f"  PV: {config.pv_capacities}")
print(f"  Battery CAPEX: {config.battery_capex}")
print(f"  Community sizes: {config.community_sizes}")
print(f"  Total combinations: {len(config.pv_capacities)} × {len(config.battery_capex)} × {len(config.community_sizes)} = {len(runner.scenario_matrix)}")


## 3. Run Scenario Sweep

Execute all scenario combinations. This may take a minute depending on number of scenarios.

In [ ]:
# Run all scenarios
print("Running scenario sweep...")
results = runner.run_all(verbose=True)
print(f"\nCompleted {len(results)} scenarios.")


## 4. Summary Results

Aggregate results into tables and key metrics.

In [ ]:
# Get summary DataFrame
summary_df = runner.summary()
print(f"Results shape: {summary_df.shape}")
print()
print("Summary statistics (battery-specific savings):")
print(summary_df[['pv_capacity_kwp', 'battery_capex_eur_kwh', 'battery_specific_savings_eur', 'payback_years', 'npv_eur']].describe())

# Display first few rows
print("\nFirst 5 scenarios:")
print(summary_df[['scenario_id', 'battery_specific_savings_eur', 'scr_pv_bess', 'ssr_pv_bess', 'payback_years']].head(10))

# Show which has best payback
best_payback = summary_df.dropna(subset=['payback_years']).nsmallest(1, 'payback_years')
print(f"\nBest payback scenario: {best_payback['scenario_id'].values[0]}")
print(f"  Payback: {best_payback['payback_years'].values[0]:.1f} years")
print(f"  Battery-specific savings: €{best_payback['battery_specific_savings_eur'].values[0]:.0f}/year")
print(f"  NPV (15 years): €{best_payback['npv_eur'].values[0]:.0f}")


## 5. Sensitivity Analysis

Create pivot tables showing battery-specific value across PV capacity and battery CAPEX.

In [ ]:
# Pivot by PV capacity and battery CAPEX
pivot_savings = summary_df.pivot_table(
    index='pv_capacity_kwp',
    columns='battery_capex_eur_kwh',
    values='battery_specific_savings_eur',
    aggfunc='first'
)

pivot_payback = summary_df.pivot_table(
    index='pv_capacity_kwp',
    columns='battery_capex_eur_kwh',
    values='payback_years',
    aggfunc='first'
)

print("Battery-specific savings (€/year):")
print(pivot_savings.round(0))
print()
print("Simple payback period (years):")
print(pivot_payback.round(1))

# Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_savings, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[0], cbar_kws={'label': '€/year'})
axes[0].set_title('Battery-Specific Annual Savings (€/year)')
axes[0].set_xlabel('Battery CAPEX (€/kWh)')
axes[0].set_ylabel('PV Capacity (kWp)')

sns.heatmap(pivot_payback, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=axes[1], cbar_kws={'label': 'years'})
axes[1].set_title('Simple Payback Period (years)')
axes[1].set_xlabel('Battery CAPEX (€/kWh)')
axes[1].set_ylabel('PV Capacity (kWp)')

plt.tight_layout()
plt.show()


## 6. Key Performance Indicators

Analyze SCR, SSR, and bill savings across scenarios.

In [ ]:
# SCR (self-consumption ratio)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# SCR improvement
summary_df['scr_improvement'] = summary_df['scr_pv_bess'] - summary_df['scr_pv_only']
axes[0].scatter(summary_df['pv_capacity_kwp'], summary_df['scr_improvement'], s=100, alpha=0.6, c=summary_df['battery_capex_eur_kwh'], cmap='viridis')
axes[0].set_xlabel('PV Capacity (kWp)')
axes[0].set_ylabel('SCR improvement (PV-BESS vs PV-only)')
axes[0].set_title('Self-Consumption Ratio Improvement')
axes[0].grid(True, alpha=0.3)

# SSR improvement
summary_df['ssr_improvement'] = summary_df['ssr_pv_bess'] - summary_df['ssr_pv_only']
scatter = axes[1].scatter(summary_df['pv_capacity_kwp'], summary_df['ssr_improvement'], s=100, alpha=0.6, c=summary_df['battery_capex_eur_kwh'], cmap='viridis')
axes[1].set_xlabel('PV Capacity (kWp)')
axes[1].set_ylabel('SSR improvement (PV-BESS vs PV-only)')
axes[1].set_title('Self-Sufficiency Ratio Improvement')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Battery CAPEX (€/kWh)')

# Battery value vs PV capacity
axes[2].scatter(summary_df['pv_capacity_kwp'], summary_df['battery_specific_savings_eur'], s=100, alpha=0.6, c=summary_df['battery_capex_eur_kwh'], cmap='viridis')
axes[2].set_xlabel('PV Capacity (kWp)')
axes[2].set_ylabel('Battery-specific savings (€/year)')
axes[2].set_title('Battery Value vs PV Size')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("KPI Summary:")
print(f"  Max SCR (PV-only): {summary_df['scr_pv_only'].max():.1%}")
print(f"  Max SCR (PV-BESS): {summary_df['scr_pv_bess'].max():.1%}")
print(f"  Max SSR (PV-only): {summary_df['ssr_pv_only'].max():.1%}")
print(f"  Max SSR (PV-BESS): {summary_df['ssr_pv_bess'].max():.1%}")
print(f"  Max energy shifted: {summary_df['energy_shifted_kwh'].max():.0f} kWh/year")


## 7. Economic Analysis

NPV and payback period analysis.

In [ ]:
# NPV by scenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NPV by PV capacity
for capex in config.battery_capex:
    subset = summary_df[summary_df['battery_capex_eur_kwh'] == capex].sort_values('pv_capacity_kwp')
    axes[0].plot(subset['pv_capacity_kwp'], subset['npv_eur'], marker='o', label=f'€{capex}/kWh')
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0].set_xlabel('PV Capacity (kWp)')
axes[0].set_ylabel('NPV (€)')
axes[0].set_title(f'Net Present Value over {config.bess_lifetime} years')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Payback period by PV capacity
for capex in config.battery_capex:
    subset = summary_df[summary_df['battery_capex_eur_kwh'] == capex].sort_values('pv_capacity_kwp')
    axes[1].plot(subset['pv_capacity_kwp'], subset['payback_years'], marker='o', label=f'€{capex}/kWh')
axes[1].set_xlabel('PV Capacity (kWp)')
axes[1].set_ylabel('Payback Period (years)')
axes[1].set_title('Simple Payback Period')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 20)

plt.tight_layout()
plt.show()

print(f"NPV Summary (discount rate {config.discount_rate:.1%}):")
positive_npv = summary_df[summary_df['npv_eur'] > 0]
print(f"  Scenarios with positive NPV: {len(positive_npv)} / {len(summary_df)}")
print(f"  Max NPV: €{summary_df['npv_eur'].max():.0f}")
print(f"  Avg payback (positive NPV only): {positive_npv['payback_years'].mean():.1f} years")


## 8. Export Results

Save summary tables for further analysis.

In [ ]:
# Export to CSV and JSON
runner.export_csv('scenario_sweep_summary.csv')
runner.export_json('scenario_sweep_full.json')

print("Exported:")
print(f"  scenario_sweep_summary.csv")
print(f"  scenario_sweep_full.json")
print()

# Summary table for paper
key_cols = ['scenario_id', 'pv_capacity_kwp', 'battery_capex_eur_kwh', 
            'battery_specific_savings_eur', 'scr_pv_only', 'scr_pv_bess',
            'ssr_pv_only', 'ssr_pv_bess', 'payback_years', 'npv_eur']
summary_df[key_cols].to_csv('scenario_sweep_key_metrics.csv', index=False)
print("Exported scenario_sweep_key_metrics.csv")

print(f"\nScenario sweep complete: {len(summary_df)} scenarios executed.")


## 9. Validation Check

Verify that dispatch simulations satisfy energy balance and bounds.

In [ ]:
# Check validation flags from first scenario
first_result = results[0]
print(f"First scenario validation: {first_result.scenario_id}")
print(f"  No-DER validation passed: {first_result.dispatch_no_der}")
print(f"  PV-only KPIs: SCR={first_result.scr_pv_only:.1%}, SSR={first_result.ssr_pv_only:.1%}")
print(f"  PV-BESS KPIs: SCR={first_result.scr_pv_bess:.1%}, SSR={first_result.ssr_pv_bess:.1%}")
print(f"  Bill (No-DER): €{first_result.bill_no_der['bill_eur']:.0f}/year")
print(f"  Bill (PV-only): €{first_result.bill_pv_only['bill_eur']:.0f}/year")
print(f"  Bill (PV-BESS): €{first_result.bill_pv_bess['bill_eur']:.0f}/year")
print(f"  Battery-specific savings: €{first_result.battery_specific_savings_eur:.0f}/year")


---\n\n## Summary\n\nThis notebook demonstrates:\n\n1. **Data preparation** — hourly load, PV, prices from real or synthetic sources\n2. **Scenario configuration** — multi-dimensional parameter sweeps\n3. **Dispatch simulation** — rule-based Chapter 4 dispatch for three scenarios\n4. **Billing calculation** — Spanish PVPC settlement with member-level cap\n5. **Economic analysis** — payback, NPV, battery-specific value\n6. **Results visualization** — sensitivity tables, heatmaps, KPI plots\n\nKey findings:\n- Battery value depends on PV size, battery cost, and price signals\n- Payback periods and NPV vary across parameter space\n- SCR/SSR improvements quantify battery contribution to self-consumption\n- Results exported for thesis reporting and further analysis\n\nNext steps:\n- Replace synthetic data with real Silver layer Parquet files\n- Extend to multiple communities and use cases\n- Add degradation modelling for extended horizons\n- Include grid constraints or ancillary service revenues (Chapter 5 extensions)\n